In [14]:
from datasets import load_dataset
import pandas as pd
import numpy as np

In [15]:
ds = load_dataset("go_emotions")

train_df = pd.DataFrame(ds["train"])
test_df = pd.DataFrame(ds["test"])

print(train_df.head())

                                                text labels       id
0  My favourite food is anything I didn't have to...   [27]  eebbqej
1  Now if he does off himself, everyone will thin...   [27]  ed00q6i
2                     WHY THE FUCK IS BAYLESS ISOING    [2]  eezlygj
3                        To make her feel threatened   [14]  ed7ypvh
4                             Dirty Southern Wankers    [3]  ed0bdzj


In [16]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)   # remove links
    text = re.sub(r'@\w+', '', text)      # remove mentions
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

train_df["clean_text"] = train_df["text"].apply(clean_text)
test_df["clean_text"] = test_df["text"].apply(clean_text)

In [17]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()

y_train = mlb.fit_transform(train_df["labels"])
y_test = mlb.transform(test_df["labels"])

print("Number of labels:", len(mlb.classes_))

Number of labels: 28


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,2)
)

X_train = vectorizer.fit_transform(train_df["clean_text"])
X_test = vectorizer.transform(test_df["clean_text"])

In [19]:
label_names = ds["train"].features["labels"].feature.names
print(label_names)

['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']


In [20]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

model = OneVsRestClassifier(
    LogisticRegression(max_iter=500, class_weight="balanced")
)
model.fit(X_train, y_train)

,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",LogisticRegre... max_iter=500)
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",None
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=

In [21]:
import numpy as np
from sklearn.metrics import f1_score

y_prob = model.predict_proba(X_test)

best_thresholds = []

for i in range(y_test.shape[1]):
    best_t = 0.5
    best_f1 = 0
    
    for t in np.arange(0.1, 0.6, 0.05):
        preds = (y_prob[:, i] >= t).astype(int)
        f1 = f1_score(y_test[:, i], preds)
        
        if f1 > best_f1:
            best_f1 = f1
            best_t = t
            
    best_thresholds.append(best_t)

print(best_thresholds)

[np.float64(0.5500000000000002), np.float64(0.5500000000000002), np.float64(0.5500000000000002), np.float64(0.5500000000000002), np.float64(0.5500000000000002), np.float64(0.5000000000000001), np.float64(0.5500000000000002), np.float64(0.5500000000000002), np.float64(0.5500000000000002), np.float64(0.5500000000000002), np.float64(0.5500000000000002), np.float64(0.5500000000000002), np.float64(0.40000000000000013), np.float64(0.5500000000000002), np.float64(0.5500000000000002), np.float64(0.5500000000000002), np.float64(0.5000000000000001), np.float64(0.5500000000000002), np.float64(0.5500000000000002), np.float64(0.40000000000000013), np.float64(0.5500000000000002), np.float64(0.4500000000000001), np.float64(0.5500000000000002), np.float64(0.5500000000000002), np.float64(0.5500000000000002), np.float64(0.5500000000000002), np.float64(0.5000000000000001), np.float64(0.4500000000000001)]


In [22]:
from sklearn.metrics import classification_report

y_prob = model.predict_proba(X_test)

threshold = 0.3

y_pred = np.zeros_like(y_prob)

for i, t in enumerate(best_thresholds):
    y_pred[:, i] = (y_prob[:, i] >= t).astype(int)

print(classification_report(
    y_test,
    y_pred,
    target_names=label_names,
    zero_division=0
))

                precision    recall  f1-score   support

    admiration       0.54      0.72      0.62       504
     amusement       0.74      0.87      0.80       264
         anger       0.32      0.56      0.41       198
     annoyance       0.23      0.43      0.30       320
      approval       0.21      0.44      0.29       351
        caring       0.19      0.50      0.28       135
     confusion       0.22      0.58      0.32       153
     curiosity       0.33      0.56      0.41       284
        desire       0.33      0.52      0.40        83
disappointment       0.18      0.37      0.24       151
   disapproval       0.23      0.53      0.32       267
       disgust       0.36      0.54      0.43       123
 embarrassment       0.23      0.38      0.29        37
    excitement       0.21      0.50      0.30       103
          fear       0.54      0.69      0.61        78
     gratitude       0.87      0.91      0.89       352
         grief       0.33      0.67      0.44  

In [10]:
from sklearn.metrics import hamming_loss

hamming_acc = 1 - hamming_loss(y_test, y_pred)
print("Hamming Accuracy:", hamming_acc)

Hamming Accuracy: 0.9447076785385243


In [23]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import LinearSVC

svm = LinearSVC()
model_s = OneVsRestClassifier(
    CalibratedClassifierCV(svm)
)
model_s.fit(X_train, y_train)

,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",CalibratedCla...r=LinearSVC())
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",None
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower 

In [25]:
from sklearn.metrics import classification_report

# 1. Get probabilities or decision scores
try:
    y_scores = model_s.decision_function(X_test)   # works for LinearSVC
except:
    y_scores = model_s.predict_proba(X_test)       # fallback (if using calibrated model)

# 2. Apply threshold (important for multi-label)
threshold = 0.3
y_pred = (y_scores >= threshold).astype(int)

# 3. Get label names (IMPORTANT FIX)
label_names = ds["train"].features["labels"].feature.names

# 4. Print report
print(classification_report(
    y_test,
    y_pred,
    target_names=label_names,
    zero_division=0
))

                precision    recall  f1-score   support

    admiration       0.64      0.57      0.60       504
     amusement       0.79      0.75      0.77       264
         anger       0.66      0.28      0.40       198
     annoyance       0.58      0.07      0.12       320
      approval       0.59      0.13      0.21       351
        caring       0.41      0.19      0.26       135
     confusion       0.54      0.24      0.33       153
     curiosity       0.47      0.23      0.31       284
        desire       0.47      0.23      0.31        83
disappointment       0.47      0.06      0.11       151
   disapproval       0.44      0.14      0.21       267
       disgust       0.64      0.30      0.41       123
 embarrassment       0.40      0.16      0.23        37
    excitement       0.59      0.23      0.33       103
          fear       0.75      0.50      0.60        78
     gratitude       0.93      0.89      0.91       352
         grief       1.00      0.17      0.29  

In [26]:
import pickle

pickle.dump(model, open("emotion_model_2.pkl", "wb"))
pickle.dump(vectorizer, open("emotion_vectorizer.pkl", "wb"))
pickle.dump(mlb, open("emotion_mlb.pkl", "wb"))